In [1]:
import os
import sys
import base64
import re
from dotenv import load_dotenv

PROJECT_PATH = os.path.join(os.getcwd(),'..')
DATA_PATH = os.path.join(PROJECT_PATH,'data')
sys.path.insert(0,PROJECT_PATH)

load_dotenv()

from src.vectorstores.qdrant_store import QdrantVectorStore
from src.ingestion.chunker import MarkdownChunker
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S"
)

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("groq").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)


/Users/albertovacas/Desktop/ai_projects/docuagent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
md_path = os.path.join(DATA_PATH, "DeepSeek_V4.md")
print("🚀 Iniciando la particion de documentos...")
chunker = MarkdownChunker()
documents = chunker.chunk_file(md_path, chunk_size=1500, chunk_overlap=300)

🚀 Iniciando la particion de documentos...


In [3]:
print("🚀 Iniciando la subida a Qdrant Cloud...")
vs = QdrantVectorStore()
vs.upload_documents(documents)

🚀 Iniciando la subida a Qdrant Cloud...


Fetching 5 files:  20%|██        | 1/5 [00:00<00:00,  5.36it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
19:21:59 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 5 files: 100%|██████████| 5/5 [00:09<00:00,  1.82s/it]
19:22:08 | INFO | src.vectorstores.qdrant_store | Qdrant collection not found. Creating collection
19:22:25 | INFO | src.vectorstores.qdrant_store | Chunks uploaded to Qdrant


In [4]:
query = "¿Qué dice el paper sobre la eficiencia del KV Cache en comparación con la versión anterior?"

# Realizar la búsqueda
results = vs.search(query, k=30)

print(f"🔍 Búsqueda: '{query}'\n")
print("-" * 50)

for i, res in enumerate(results):
    print(f"📍 Resultado {i+1} (Score: {res.score:.4f})")
    print(f"📄 Ubicación: {res.metadata.get('page', 'Desconocida')}")
    print(f"📝 Texto:\n{res.text}")
    print("-" * 50)

🔍 Búsqueda: '¿Qué dice el paper sobre la eficiencia del KV Cache en comparación con la versión anterior?'

--------------------------------------------------
📍 Resultado 1 (Score: 0.5226)
📄 Ubicación: 1
📝 Texto:
Figure 1 | Left: benchmark performance of DeepSeek-V4-Pro-Max and its counterparts. Right: inference FLOPs and KV cache size of DeepSeek-V4 series and DeepSeek-V3.2.
- **Tipo de gráfico:** Barras
- **Título:** Rendimiento comparativo de DeepSeek-V4-Pro-Max y series DeepSeek-V4 y DeepSeek-V3.2.
--------------------------------------------------
📍 Resultado 2 (Score: 0.5213)
📄 Ubicación: 23
📝 Texto:
within that prefix. Despite computational zero-redundancy, this strategy is inefficient for
modern SSD-based storage systems — only a small subset of the stored SWA KV cache
will be accessed for each hitting request, which leads to an unbalanced write-intensive
access pattern.
• Periodic Checkpointing.This strategy checkpoints SWA KV entries of the last 𝑛win tokens
within every 𝑝 toke